In [2]:
#데이터 불러오기 및 원본 데이터 확인
import pandas as pd

df = pd.read_json("team2_data.json")

print("원본 데이터 크기:", df.shape)
display(df.head())

원본 데이터 크기: (121, 11)


,cve_id,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description,service,version
0,CVE-2011-2523,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-78,vsftpd 2.3.4 downloaded between 20110630 and 2...,ftp,2.3.4
1,CVE-2010-4478,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,"OpenSSH 5.6 and earlier, when J-PAKE is enable...",ssh,4.7p1 Debian 8ubuntu1
2,CVE-2016-1908,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,The client in OpenSSH before 7.2 mishandles fa...,ssh,4.7p1 Debian 8ubuntu1
3,CVE-2015-5600,8.1,HIGH,NETWORK,HIGH,NONE,NONE,CWE-264,The kbdint_next_device function in auth2-chall...,ssh,4.7p1 Debian 8ubuntu1
4,CVE-2015-8325,7.8,HIGH,LOCAL,LOW,LOW,NONE,CWE-1262,The do_setup_env function in session.c in sshd...,ssh,4.7p1 Debian 8ubuntu1


In [3]:
#데이터 구조 및 결측치 확인
print(df.info())

print("\n컬럼 목록")
print(df.columns.tolist())

print("\n결측치")
print(df.isnull().sum())

print("\n위험도 분포")
print(df["severity"].value_counts(dropna=False))

print("\n중복 CVE")
print(df.duplicated(subset=["cve_id"]).sum())

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 11 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   cve_id               121 non-null    str    
 1   cvss_score           121 non-null    float64
 2   severity             121 non-null    str    
 3   attack_vector        121 non-null    str    
 4   attack_complexity    121 non-null    str    
 5   privileges_required  121 non-null    str    
 6   user_interaction     121 non-null    str    
 7   cwe                  121 non-null    str    
 8   description          121 non-null    str    
 9   service              121 non-null    str    
 10  version              121 non-null    str    
dtypes: float64(1), str(10)
memory usage: 10.5 KB
None

컬럼 목록
['cve_id', 'cvss_score', 'severity', 'attack_vector', 'attack_complexity', 'privileges_required', 'user_interaction', 'cwe', 'description', 'service', 'version']

결측치
cve_id                 0
cv

In [4]:
#데이터 품질 및 클래스 분포 확인
print("=" * 50)
print("데이터 품질 검사")
print("=" * 50)

print(f"데이터 개수 : {len(df)}")
print(f"컬럼 개수 : {len(df.columns)}")

print("\n컬럼 정보")
print(df.dtypes)

print("\n결측치")
print(df.isnull().sum())

print("\n중복 CVE")
print(df.duplicated(subset=["cve_id"]).sum())

print("\nSeverity 분포")
print(df["severity"].value_counts())

print("\nCVSS 통계")
print(df["cvss_score"].describe())

데이터 품질 검사
데이터 개수 : 121
컬럼 개수 : 11

컬럼 정보
cve_id                     str
cvss_score             float64
severity                   str
attack_vector              str
attack_complexity          str
privileges_required        str
user_interaction           str
cwe                        str
description                str
service                    str
version                    str
dtype: object

결측치
cve_id                 0
cvss_score             0
severity               0
attack_vector          0
attack_complexity      0
privileges_required    0
user_interaction       0
cwe                    0
description            0
service                0
version                0
dtype: int64

중복 CVE
0

Severity 분포
severity
MEDIUM      65
HIGH        35
LOW         17
CRITICAL     4
Name: count, dtype: int64

CVSS 통계
count    121.000000
mean       5.730579
std        1.913716
min        1.200000
25%        4.200000
50%        5.300000
75%        7.300000
max       10.000000
Name: cvss_score, dtype:

In [5]:
#데이터 전처리 함수 정의
def preprocess_cve_data(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()

    # 컬럼명 정리
    data.columns = (
        data.columns
        .str.strip()
        .str.lower()
    )

    required_columns = [
        "cve_id",
        "cvss_score",
        "severity",
        "attack_vector",
        "attack_complexity",
        "privileges_required",
        "user_interaction",
        "cwe",
        "description",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"필수 컬럼이 없습니다: {missing_columns}"
        )

    # CVE ID 중복 제거
    data = data.drop_duplicates(
        subset=["cve_id"],
        keep="first"
    )

    # CVSS 숫자 변환
    data["cvss_score"] = pd.to_numeric(
        data["cvss_score"],
        errors="coerce"
    )

    # 핵심 데이터 결측 행 제거
    data = data.dropna(
        subset=[
            "cve_id",
            "cvss_score",
            "severity",
            "attack_vector",
            "attack_complexity",
            "privileges_required",
            "user_interaction",
        ]
    )

    # 문자열 데이터 통일
    category_columns = [
        "severity",
        "attack_vector",
        "attack_complexity",
        "privileges_required",
        "user_interaction",
        "cwe",
    ]

    for column in category_columns:
        data[column] = (
            data[column]
            .fillna("UNKNOWN")
            .astype(str)
            .str.strip()
            .str.upper()
        )

    # 설명 결측치 처리
    data["description"] = (
        data["description"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # CVSS 정상 범위만 유지
    data = data[
        data["cvss_score"].between(0, 10)
    ]

    # 위험도 NONE 6건 제거
    valid_severity = [
        "LOW",
        "MEDIUM",
        "HIGH",
        "CRITICAL",
    ]

    data = data[
        data["severity"].isin(valid_severity)
    ]

    return data.reset_index(drop=True)

In [6]:
#전처리 실행 및 결과 확인
clean_df = preprocess_cve_data(df)

print("전처리 전:", df.shape)
print("전처리 후:", clean_df.shape)

print("\n결측치")
print(clean_df.isnull().sum())

print("\n위험도 분포")
print(clean_df["severity"].value_counts())

display(clean_df.head())

전처리 전: (121, 11)
전처리 후: (121, 11)

결측치
cve_id                 0
cvss_score             0
severity               0
attack_vector          0
attack_complexity      0
privileges_required    0
user_interaction       0
cwe                    0
description            0
service                0
version                0
dtype: int64

위험도 분포
severity
MEDIUM      65
HIGH        35
LOW         17
CRITICAL     4
Name: count, dtype: int64


,cve_id,cvss_score,severity,attack_vector,attack_complexity,privileges_required,user_interaction,cwe,description,service,version
0,CVE-2011-2523,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-78,vsftpd 2.3.4 downloaded between 20110630 and 2...,ftp,2.3.4
1,CVE-2010-4478,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,"OpenSSH 5.6 and earlier, when J-PAKE is enable...",ssh,4.7p1 Debian 8ubuntu1
2,CVE-2016-1908,9.8,CRITICAL,NETWORK,LOW,NONE,NONE,CWE-287,The client in OpenSSH before 7.2 mishandles fa...,ssh,4.7p1 Debian 8ubuntu1
3,CVE-2015-5600,8.1,HIGH,NETWORK,HIGH,NONE,NONE,CWE-264,The kbdint_next_device function in auth2-chall...,ssh,4.7p1 Debian 8ubuntu1
4,CVE-2015-8325,7.8,HIGH,LOCAL,LOW,LOW,NONE,CWE-1262,The do_setup_env function in session.c in sshd...,ssh,4.7p1 Debian 8ubuntu1


In [7]:
#cvss 이상치 검사
print("=" * 50)
print("이상치 검사")
print("=" * 50)

invalid_cvss = clean_df[
    ~clean_df["cvss_score"].between(0, 10)
]

print(f"CVSS 이상치 : {len(invalid_cvss)}")

valid_vectors = [
    "NETWORK",
    "LOCAL",
    "PHYSICAL",
    "ADJACENT_NETWORK"
]

invalid_vector = clean_df[
    ~clean_df["attack_vector"].isin(valid_vectors)
]

print(f"Attack Vector 이상치 : {len(invalid_vector)}")

valid_complexity = [
    "LOW",
    "HIGH"
]

invalid_complexity = clean_df[
    ~clean_df["attack_complexity"].isin(valid_complexity)
]

print(f"Attack Complexity 이상치 : {len(invalid_complexity)}")

이상치 검사
CVSS 이상치 : 0
Attack Vector 이상치 : 0
Attack Complexity 이상치 : 27


In [8]:
#클래스 불균형 확인
print("=" * 50)
print("클래스 불균형 확인")
print("=" * 50)

severity_counts = clean_df["severity"].value_counts()

print(severity_counts)

ratio = severity_counts.max() / severity_counts.min()

print(f"\n최대/최소 클래스 비율 : {ratio:.2f}")

if ratio > 5:
    print("⚠ 클래스 불균형이 존재합니다.")
else:
    print("✓ 클래스 분포가 비교적 균형적입니다.")

클래스 불균형 확인
severity
MEDIUM      65
HIGH        35
LOW         17
CRITICAL     4
Name: count, dtype: int64

최대/최소 클래스 비율 : 16.25
⚠ 클래스 불균형이 존재합니다.


In [9]:
# 전처리 데이터 저장
clean_df.to_csv(
    "cve_preprocessed.csv",
    index=False,
    encoding="utf-8-sig"
)

print("cve_preprocessed.csv 저장 완료")

cve_preprocessed.csv 저장 완료


In [10]:
# 전처리 결과 리포트 생성
report = f"""
===============================
Preprocessing Report
===============================

원본 데이터 수
: {len(df)}

전처리 후 데이터 수
: {len(clean_df)}

중복 제거
: {df.duplicated(subset=['cve_id']).sum()}

결측치(CWE)
: {df['cwe'].isnull().sum()}

CVSS 이상치
: {len(invalid_cvss)}

Attack Vector 이상치
: {len(invalid_vector)}

Attack Complexity 이상치
: {len(invalid_complexity)}

Severity 분포

{severity_counts.to_string()}

===============================
"""

print(report)

with open(
    "preprocessing_report.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

print("preprocessing_report.txt 생성 완료")


Preprocessing Report

원본 데이터 수
: 121

전처리 후 데이터 수
: 121

중복 제거
: 0

결측치(CWE)
: 0

CVSS 이상치
: 0

Attack Vector 이상치
: 0

Attack Complexity 이상치
: 27

Severity 분포

severity
MEDIUM      65
HIGH        35
LOW         17
CRITICAL     4


preprocessing_report.txt 생성 완료


In [20]:
#클래스 불균형 확인용 이유는 마지막 evaluation에서 확인할 것.
#print(df["severity"].value_counts())
#print(df["severity"].value_counts(normalize=True))

<<핵심 과정>>
1. 원본 CVE 데이터 불러오기
2. 컬럼명 정리 및 결측치 처리
3. 중복 제거 및 데이터 정제
4. CVSS 이상치 확인
5. 클래스 분포 확인
6. 전처리 완료 데이터 저장

<<결과>>
=> 머신러닝 학습이 가능한 형태의 데이터셋을 생성하고 전처리 결과를 저장하는 파일 작업 완료!!
